In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

class Generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, output_dim), nn.Tanh())
    def forward(self, z):
        return self.model(z)

class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(nn.Linear(input_dim, 512), nn.LeakyReLU(0.2), nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, img):
        return self.model(img)

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
mnist = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(mnist, batch_size=64, shuffle=True)
latent_dim = 100
img_dim = 28 * 28
lr = 0.0002
num_epochs = 50
generator = Generator(latent_dim, img_dim)
discriminator = Discriminator(img_dim)
optimizer_G = optim.Adam(generator.parameters(), lr=lr)
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)
criterion = nn.BCELoss()
for epoch in range(num_epochs):
    for i, (imgs, _) in enumerate(dataloader):
        real_labels = torch.ones(imgs.size(0), 1)
        fake_labels = torch.zeros(imgs.size(0), 1)
        optimizer_D.zero_grad()
        real_imgs = imgs.view(imgs.size(0), -1)
        outputs = discriminator(real_imgs)
        d_loss_real = criterion(outputs, real_labels)
        d_loss_real.backward()
        z = torch.randn(imgs.size(0), latent_dim)
        fake_imgs = generator(z)
        outputs = discriminator(fake_imgs.detach())
        d_loss_fake = criterion(outputs, fake_labels)
        d_loss_fake.backward()
        optimizer_D.step()
        optimizer_G.zero_grad()
        outputs = discriminator(fake_imgs)
        g_loss = criterion(outputs, real_labels)
        g_loss.backward()
        optimizer_G.step()
    print(f'Epoch [{epoch+1}/{num_epochs}], d_loss: {d_loss_real.item() +d_loss_fake.item():.4f}, g_loss: {g_loss.item():.4f}')
z = torch.randn(16, latent_dim)
fake_imgs = generator(z).view(-1, 1, 28, 28).detach().numpy()
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i in range(16):
    axes[i // 4, i % 4].imshow(fake_imgs[i][0], cmap='gray')
    axes[i // 4, i % 4].axis('off')
plt.show()

Epoch [1/50], d_loss: 0.5362, g_loss: 7.0178
Epoch [2/50], d_loss: 1.2951, g_loss: 2.7856
Epoch [3/50], d_loss: 0.0902, g_loss: 4.0175
Epoch [4/50], d_loss: 0.3239, g_loss: 4.0259
Epoch [5/50], d_loss: 0.1079, g_loss: 5.0351
Epoch [6/50], d_loss: 0.6038, g_loss: 3.6494
Epoch [7/50], d_loss: 1.0237, g_loss: 3.0319
Epoch [8/50], d_loss: 0.3043, g_loss: 5.1879
Epoch [9/50], d_loss: 0.1819, g_loss: 4.2522
Epoch [10/50], d_loss: 0.2631, g_loss: 4.4163
Epoch [11/50], d_loss: 0.2793, g_loss: 3.4562
Epoch [12/50], d_loss: 0.2215, g_loss: 5.2293
Epoch [13/50], d_loss: 0.6430, g_loss: 2.6018
Epoch [14/50], d_loss: 0.2625, g_loss: 3.5763
Epoch [15/50], d_loss: 0.2424, g_loss: 4.9131
Epoch [16/50], d_loss: 0.3115, g_loss: 3.9164
Epoch [17/50], d_loss: 0.5066, g_loss: 3.3151
Epoch [18/50], d_loss: 0.1540, g_loss: 3.4250
Epoch [19/50], d_loss: 0.5361, g_loss: 3.3589
Epoch [20/50], d_loss: 0.5706, g_loss: 2.2580
Epoch [21/50], d_loss: 0.4627, g_loss: 3.0569
Epoch [22/50], d_loss: 0.4805, g_loss: 3.53